# Aykırı Değerler (Outliers)
### Amaç:
  1. Veri setindeki aykırı değerleri tespit ve inceleme
  2. Aykırı değerleri temizleyip güncel veri setini oluştur

### Adımlar:
  1. Veri setinin yüklenmesi
  2. Sayısal sütunlar belirlenir
  3. Sayısal sütunların özet istatistikleri
  4. iqr ile aykırı değer sınırlarını hesapla
  5. Aykırı değer içeren kayıtları tespit et
  6. Aykırı değerleri sütun bazında incele
  7. Aykırı değerleri temizle
  8. Temizlenmiş veri setinin kaydedilmesi

In [4]:
import pandas as pd

In [67]:
# 1. veri setini yükle
dataFrame = pd.read_csv("e_ticaret_veri_seti_tekrarlayan_kayitlar_duzenlendi.csv")

In [69]:
# 2. sayısal sütunların tespit edilmesi
sayisal_sutunlar = dataFrame.select_dtypes(include = ["float64","int64"]).columns.tolist()

In [71]:
sayisal_sutunlar

['adet',
 'birim_fiyat',
 'indirim_orani',
 'kargo_ucreti',
 'teslimat_gunu',
 'musteri_puani',
 'kar_marji_orani',
 'toplam_tutar',
 'siparis_yili',
 'siparis_ay',
 'siparis_gunu']

In [73]:
# 3. özet istatistik
dataFrame.describe()

,adet,birim_fiyat,indirim_orani,kargo_ucreti,teslimat_gunu,musteri_puani,kar_marji_orani,toplam_tutar,siparis_yili,siparis_ay,siparis_gunu
count,1342.000000,1342.000000,1326.000000,1342.000000,1342.000000,1342.000000,1342.000000,1342.000000,1342.000000,1342.000000,1342.000000
mean,2.122206,669.145961,0.052413,39.743294,3.374814,3.989568,0.302756,1380.023666,2024.533532,6.397914,15.638599
std,1.944793,1067.583282,0.060234,14.168266,1.837644,1.097708,0.069967,2996.499891,0.499060,3.461250,8.712994
min,1.000000,93.660000,0.000000,0.000000,1.000000,0.000000,0.180000,89.680000,2024.000000,1.000000,1.000000
25%,1.000000,199.795000,0.000000,29.900000,2.000000,3.000000,0.244000,323.405000,2024.000000,3.000000,8.000000
50%,2.000000,287.735000,0.050000,39.900000,3.000000,4.000000,0.305000,666.535000,2025.000000,6.000000,15.000000
75%,3.000000,830.052500,0.100000,49.900000,4.000000,5.000000,0.362000,1423.067500,2025.000000,9.000000,23.000000
max,38.000000,18145.230000,0.200000,59.900000,31.000000,5.000000,0.420000,63870.200000,2025.000000,12.000000,31.000000


In [75]:
# 4. iqr ile aykırı değer tespiti
aykiri_sinirlar = {}
for sutun in sayisal_sutunlar:
    q1 = dataFrame[sutun].quantile(0.25)
    q3 = dataFrame[sutun].quantile(0.75)
    
    iqr = q3 - q1
    
    alt_sinir = q1 - 1.5 * iqr

    ust_sinir = q3 + 1.5 * iqr
    

    aykiri_sinirlar[sutun] = {
      "q1":q1,
      "q3":q3,
      "iqr":iqr,
      "alt_sinir":alt_sinir,
      "ust_sinir":ust_sinir
  }

In [77]:
aykiri_sinirlar

{'adet': {'q1': 1.0,
  'q3': 3.0,
  'iqr': 2.0,
  'alt_sinir': -2.0,
  'ust_sinir': 6.0},
 'birim_fiyat': {'q1': 199.79500000000002,
  'q3': 830.0525,
  'iqr': 630.2574999999999,
  'alt_sinir': -745.59125,
  'ust_sinir': 1775.4387499999998},
 'indirim_orani': {'q1': 0.0,
  'q3': 0.1,
  'iqr': 0.1,
  'alt_sinir': -0.15000000000000002,
  'ust_sinir': 0.25},
 'kargo_ucreti': {'q1': 29.9,
  'q3': 49.9,
  'iqr': 20.0,
  'alt_sinir': -0.10000000000000142,
  'ust_sinir': 79.9},
 'teslimat_gunu': {'q1': 2.0,
  'q3': 4.0,
  'iqr': 2.0,
  'alt_sinir': -1.0,
  'ust_sinir': 7.0},
 'musteri_puani': {'q1': 3.0,
  'q3': 5.0,
  'iqr': 2.0,
  'alt_sinir': 0.0,
  'ust_sinir': 8.0},
 'kar_marji_orani': {'q1': 0.244,
  'q3': 0.362,
  'iqr': 0.118,
  'alt_sinir': 0.067,
  'ust_sinir': 0.5389999999999999},
 'toplam_tutar': {'q1': 323.405,
  'q3': 1423.0674999999999,
  'iqr': 1099.6625,
  'alt_sinir': -1326.08875,
  'ust_sinir': 3072.5612499999997},
 'siparis_yili': {'q1': 2024.0,
  'q3': 2025.0,
  'iqr': 1.

In [104]:
# 5. aykırı değer içeren kayıtları tespit edelim
aykiri_maskesi = pd.Series(False, index=dataFrame.index)

for sutun in sayisal_sutunlar:
    alt_sinir = aykiri_sinirlar[sutun]["alt_sinir"]
    ust_sinir = aykiri_sinirlar[sutun]["ust_sinir"]

    sutun_maskesi = (dataFrame[sutun] < alt_sinir) | (dataFrame[sutun] > ust_sinir)
    aykiri_maskesi = aykiri_maskesi | sutun_maskesi





In [102]:
aykiri_kayitlar = dataFrame[aykiri_maskesi]


In [106]:
aykiri_kayitlar.head()

,siparis_id,musteri_id,siparis_tarihi,sehir,bolge,kategori,urun_adi,adet,birim_fiyat,indirim_orani,...,musteri_tipi,teslimat_gunu,musteri_puani,iade_durumu,kar_marji_orani,toplam_tutar,siparis_yili,siparis_ay,siparis_gunu,haftanin_gunu
7,SIP101172,MUS1357,2024-07-22 13:26:00,Ankara,İç Anadolu,Ofis,Mekanik Klavye,3,2344.47,0.00,...,Mevcut,3.0,4.0,Hayır,0.312,7093.31,2024,7,22,Monday
8,SIP100779,MUS1354,2025-12-05 11:19:00,İstanbul,Marmara,Kitap,Python Notları,4,219.14,0.00,...,Yeni,18.0,5.0,Evet,0.408,906.46,2025,12,5,Friday
15,SIP100598,MUS1223,2025-08-08 19:03:00,İstanbul,Marmara,Elektronik,Akıllı Saat,3,2889.29,0.00,...,Yeni,1.0,1.0,Hayır,0.189,8707.77,2025,8,8,Friday
18,SIP101277,MUS1287,2024-07-31 17:52:00,Konya,İç Anadolu,Spor,Dambıl Seti,2,1618.73,0.05,...,Yeni,3.0,5.0,Hayır,0.208,3105.49,2024,7,31,Wednesday
23,SIP101061,MUS1294,2024-05-04 15:41:00,İstanbul,Marmara,Elektronik,Kablosuz Kulaklık,3,1445.43,0.00,...,Mevcut,4.0,4.0,Evet,0.407,4376.19,2024,5,4,Saturday


In [108]:
# 6. aykırı değer sayılarını sutun bazında inceleyelim
for sutun in sayisal_sutunlar:
    alt_sinir = aykiri_sinirlar[sutun]["alt_sinir"]
    ust_sinir = aykiri_sinirlar[sutun]["ust_sinir"]
    
    aykiri_sayi = ((dataFrame[sutun] < alt_sinir ) | (dataFrame[sutun] > ust_sinir)).sum()
    print(f"{sutun} : {aykiri_sayi}")

adet : 5
birim_fiyat : 90
indirim_orani : 0
kargo_ucreti : 0
teslimat_gunu : 6
musteri_puani : 0
kar_marji_orani : 0
toplam_tutar : 120
siparis_yili : 0
siparis_ay : 0
siparis_gunu : 0


In [110]:
# 7. aykırı değer içeren kayıtları veri setinden temizleyelim
dataFrame_temiz = dataFrame[~aykiri_maskesi].copy()

dataFrame_temiz.shape

(1186, 22)

In [112]:
dataFrame_temiz.to_csv("e_ticaret_veri_seti_aykiri_degerler_duzenlendi.csv",index = False, encoding = "utf-8-sig")